In [1]:
import pathlib as pl
import pandas as pd
import zipfile
import duckdb

In [32]:
file_folder_path = pl.Path("C:/Users/gunja/OneDrive/Documents/ACTIVE PROJECTS/conflict_pred/data/raw/GDLET_ZIP")

all_zip_files = list(file_folder_path.glob("*.zip"))

if not all_zip_files:
    print(f"No file exists in the folder: {file_folder_path}")
    raise SystemExit(0)

In [33]:
all_zip_files

[WindowsPath('C:/Users/gunja/OneDrive/Documents/ACTIVE PROJECTS/conflict_pred/data/raw/GDLET_ZIP/200601.zip')]

In [19]:
# Define exact 0-based column indices to keep
SELECTED_COLS = [0, 1, 3, 7, 12, 17, 22, 28, 30, 33, 34]

# Define corresponding clean column names
COLUMN_NAMES = [
    "statement_id",
    "timestamp",
    "year",
    "country_a",
    "source_type",
    "country_b",
    "target_type",
    "event_root_code",
    "goldstein_scale",
    "num_articles",
    "avg_tone",
]

Indices

0 : GlobalEnventID

1 : Time

3 : Year

7 : Country_a

12: Source_type

17: Country_b

22: Target_type

28: EvenRootCode -> CAMEO CODE

30: Goldstein_Scale

33: Number of Articles

34: Sentiment Score

53: Action_Coutry_Code: Location where the event occured.

In [43]:
for zip_file_path in all_zip_files:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_file:
        csv_files = zip_file.namelist()
        for csv in csv_files:
            with zip_file.open(csv) as file:  
                df = pd.read_csv(
                    file,
                    sep="\t",
                    header=None,
                    usecols=SELECTED_COLS,
                    names=COLUMN_NAMES,
                    low_memory=False,
                )
                print(df)
                

        statement_id  timestamp  year country_a source_type country_b  \
0          175632684   20060101  2006       NaN         NaN       AFG   
1          175632685   20060101  2006       NaN         NaN       AFG   
2          175632686   20060101  2006       NaN         NaN       AFG   
3          175632687   20060101  2006       NaN         NaN       AFG   
4          175632688   20060101  2006       NaN         NaN       AFG   
...              ...        ...   ...       ...         ...       ...   
314879     175947563   20060131  2006       NaN         REB       LKA   
314880     175947564   20060131  2006       NaN         GOV       KEN   
314881     175947565   20060131  2006       NaN         GOV       KEN   
314882     175947566   20060131  2006       NaN         GOV       UGA   
314883     175947567   20060131  2006       NaN         NaN       NaN   

       target_type  event_root_code  goldstein_scale  num_articles  avg_tone  
0              NaN                4         

In [39]:
df_grouped = (
    df.groupby(["year", "country_a", "country_b", "event_root_code"])
    .apply(
        lambda x: pd.Series(
            {
                "weighted_avg_tone": (x["num_articles"] * x["avg_tone"]).sum()
                / x["num_articles"].sum(),
                "mean_goldstein": x[
                    "goldstein_scale"
                ].mean(),  # Average impact score
            }
        )
    )
    .reset_index()
)

print(df_grouped)

       year country_a country_b  event_root_code  weighted_avg_tone  \
0      2000       ABW       ABW             16.0           2.238806   
1      2000       ABW       CRI              4.0           2.781215   
2      2000       ABW       LBR              4.0           4.885139   
3      2000       ABW       NLD             16.0           5.579399   
4      2000       ABW       PER              4.0           3.302062   
...     ...       ...       ...              ...                ...   
77953  2000       ZWE       ZWE             15.0           4.333499   
77954  2000       ZWE       ZWE             16.0           5.036053   
77955  2000       ZWE       ZWE             17.0           4.181428   
77956  2000       ZWE       ZWE             18.0           3.984548   
77957  2000       ZWE       ZWE             19.0           4.261337   

       mean_goldstein  
0           -4.000000  
1            2.650000  
2            2.800000  
3           -4.000000  
4            2.800000  
...

Inside GDLET folder each parquet file is presenting data form one particular year. I want data from each in in one single parquet file combined. For this i I have to do some operation on existing files.

In [17]:
new_df = duckdb.sql("""
SELECT 
    year || '_' || LEAST(country_a, country_b) || '_' || GREATEST(country_a, country_b) AS dyad,
    *
FROM '../data/interim/GDELT/2000.parquet'
WHERE country_a != country_b
"""
).df()


In [18]:
new_df

,dyad,year,country_a,country_b,event_root_code,mean_goldstein,weighted_avg_tone
0,2000_ABW_CRI,2000,ABW,CRI,4.0,2.650000,2.781215
1,2000_ABW_LBR,2000,ABW,LBR,4.0,2.800000,4.885139
2,2000_ABW_NLD,2000,ABW,NLD,16.0,-4.000000,5.579399
3,2000_ABW_PER,2000,ABW,PER,4.0,2.800000,3.302062
4,2000_ABW_USA,2000,ABW,USA,3.0,4.000000,5.579399
...,...,...,...,...,...,...,...
74820,2000_ZMB_ZWE,2000,ZWE,ZMB,12.0,-4.000000,6.109912
74821,2000_ZMB_ZWE,2000,ZWE,ZMB,13.0,-4.400000,4.136219
74822,2000_ZMB_ZWE,2000,ZWE,ZMB,15.0,-7.200000,1.742160
74823,2000_ZMB_ZWE,2000,ZWE,ZMB,17.0,-5.000000,2.764977


In [19]:
new_df['dyad'].is_unique

False

Dyad column is not unique for event_root_code

In [20]:
ZMB_ZWE = new_df[new_df['dyad'] == '2000_ZMB_ZWE']
ZMB_ZWE

,dyad,year,country_a,country_b,event_root_code,mean_goldstein,weighted_avg_tone
74341,2000_ZMB_ZWE,2000,ZMB,ZWE,1.0,-0.160000,4.570625
74342,2000_ZMB_ZWE,2000,ZMB,ZWE,2.0,3.070588,4.957295
74343,2000_ZMB_ZWE,2000,ZMB,ZWE,3.0,4.300000,6.906360
74344,2000_ZMB_ZWE,2000,ZMB,ZWE,4.0,2.584146,5.922557
74345,2000_ZMB_ZWE,2000,ZMB,ZWE,5.0,6.096552,7.492828
74346,2000_ZMB_ZWE,2000,ZMB,ZWE,6.0,6.400000,4.011870
74347,2000_ZMB_ZWE,2000,ZMB,ZWE,7.0,7.664706,4.152606
74348,2000_ZMB_ZWE,2000,ZMB,ZWE,8.0,7.000000,6.143448
74349,2000_ZMB_ZWE,2000,ZMB,ZWE,10.0,-5.000000,4.655291
74350,2000_ZMB_ZWE,2000,ZMB,ZWE,11.0,-2.000000,4.166164


In [17]:
l = new_df['event_root_code'].unique().tolist()
l

[4.0,
 16.0,
 3.0,
 2.0,
 19.0,
 1.0,
 5.0,
 11.0,
 17.0,
 7.0,
 8.0,
 12.0,
 10.0,
 6.0,
 13.0,
 14.0,
 18.0,
 9.0,
 15.0,
 20.0]

because of the event root code column the primary key having vertical replicaition of data.

In [ ]:
input_pattern = "..\data\interim\GDELT\*.parquet"
output_parquet = "..\data\interim\gdlet_filtered*.parquet"

query = f"""
WITH filtered_data AS (
    SELECT 
        year || '_' || LEAST(country_a, country_b) || '_' || GREATEST(country_a, country_b) AS dyad,
        year,
        LEAST(country_a, country_b) AS country_a,
        GREATEST(country_a, country_b) AS country_b,
        event_root_code AS event_code,
        mean_goldstein,
        weighted_avg_tone
    FROM '{input_pattern}'
    WHERE country_a != country_b
)
PIVOT filtered_data
ON event_code IN (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20)
USING 
    AVG(weighted_avg_tone) AS 'tone_code_{{}}',
    AVG(mean_goldstein) AS 'goldstein_code_{{}}'
GROUP BY dyad, year, country_a, country_b
ORDER BY year, country_a, country_b;
"""

con = duckdb.connect()

# Load into memory as Pandas DataFrame
new_df_main = con.sql(query).df()

# Or export straight to compressed Parquet
#con.execute(
 #   f"COPY ({query}) TO '{output_parquet}' (FORMAT PARQUET, COMPRESSION 'ZSTD');"
#)
print("Pipeline complete! Flat, clean dataset generated successfully.")
new_df_main

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\gunja\AppData\Local\Temp\ipykernel_5568\4261018148.py:1: SyntaxWarning: invalid escape sequence '\d'
  input_pattern = "..\data\interim\GDELT\*.parquet"


Pipeline complete! Flat, clean dataset generated successfully.


,dyad,year,country_a,country_b,1_tone_code_{},1_goldstein_code_{},2_tone_code_{},2_goldstein_code_{},3_tone_code_{},3_goldstein_code_{},...,16_tone_code_{},16_goldstein_code_{},17_tone_code_{},17_goldstein_code_{},18_tone_code_{},18_goldstein_code_{},19_tone_code_{},19_goldstein_code_{},20_tone_code_{},20_goldstein_code_{}
0,1920_AFG_AFR,1920,AFG,AFR,-3.039751,0.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1920_AFG_AUS,1920,AFG,AUS,-2.612693,0.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-8.091286,-10.000000,NaN,NaN
2,1920_AFG_BGD,1920,AFG,BGD,-3.652968,0.000000,-5.605632,3.000000,-5.333333,4.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1920_AFG_BTN,1920,AFG,BTN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1920_AFG_CAN,1920,AFG,CAN,-1.571165,0.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-1.461632,-10.000000,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270520,2021_YEM_ZAF,2021,YEM,ZAF,-0.592593,0.000000,NaN,NaN,-4.860847,5.200000,...,NaN,NaN,-4.591368,-5.000000,NaN,NaN,-0.255505,-10.000000,NaN,NaN
270521,2021_YEM_ZWE,2021,YEM,ZWE,-4.992604,-0.150000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
270522,2021_ZAF_ZMB,2021,ZAF,ZMB,-0.285846,0.350877,NaN,NaN,-1.491102,4.000000,...,-1.496169,-4.000000,-1.331596,-5.000000,NaN,NaN,-4.618246,-10.000000,NaN,NaN
270523,2021_ZAF_ZWE,2021,ZAF,ZWE,-3.960427,0.114074,-2.445329,2.757353,-2.002024,4.260000,...,-1.564004,-5.230769,-5.590073,-5.305455,-6.849124,-9.333333,-5.119687,-9.853659,NaN,NaN


In [13]:
new_df_main.columns.tolist()

['dyad',
 'year',
 'country_a',
 'country_b',
 '1_tone',
 '1_goldstein',
 '2_tone',
 '2_goldstein',
 '3_tone',
 '3_goldstein',
 '4_tone',
 '4_goldstein',
 '5_tone',
 '5_goldstein',
 '6_tone',
 '6_goldstein',
 '7_tone',
 '7_goldstein',
 '8_tone',
 '8_goldstein',
 '9_tone',
 '9_goldstein',
 '10_tone',
 '10_goldstein',
 '11_tone',
 '11_goldstein',
 '12_tone',
 '12_goldstein',
 '13_tone',
 '13_goldstein',
 '14_tone',
 '14_goldstein',
 '15_tone',
 '15_goldstein',
 '16_tone',
 '16_goldstein',
 '17_tone',
 '17_goldstein',
 '18_tone',
 '18_goldstein',
 '19_tone',
 '19_goldstein',
 '20_tone',
 '20_goldstein']

In [14]:
new_df_main['dyad'].is_unique

True

dyadas are now unique

In [15]:
result_1 = new_df_main[new_df_main['dyad'] == '2000_ZMB_ZWE']
result_1[['12_tone', '12_goldstein', '13_tone', '13_goldstein', '14_tone', '14_goldstein', '15_tone', '15_goldstein', '16_tone', '16_goldstein', '17_tone', '17_goldstein', '18_tone', '18_goldstein', '19_tone', '19_goldstein', '20_tone', '20_goldstein']]

,12_tone,12_goldstein,13_tone,13_goldstein,14_tone,14_goldstein,15_tone,15_goldstein,16_tone,16_goldstein,17_tone,17_goldstein,18_tone,18_goldstein,19_tone,19_goldstein,20_tone,20_goldstein
8350,6.660725,-4.0,3.702151,-4.4,NaN,NaN,1.74216,-7.2,6.303725,-4.0,2.764977,-5.0,5.555556,-9.0,3.675872,-9.677778,NaN,NaN
